# Stage 10 Phase 1 — LSTM residual pretrain (Colab)

Smart App Control blocks PyTorch on the dev machine, so training runs here and the
checkpoints come back (option C). Inference on the dev machine is pure numpy (option D).

**Before running:** push a branch with the `stage10_*` files **and**
`models/stage10/oof_preds.csv` committed (or upload the CSV in cell 2).

Outputs to download at the end: `pretrain_<season>.npz` (numpy weights — the runtime
path), `pretrain_<season>.pt`, `stage10_config.json`, `stage10_calibration.json`.

Runtime: CPU is fine (~15–30 min for 5 folds). Set Runtime→CPU for determinism.

In [ ]:
# 1 · clone + deps
REPO = 'https://github.com/Andrej-Andonovski/Andrej_dpl_komar_ai.git'
BRANCH = 'stage10-phase1'   # <-- the branch you pushed the stage10 files on
import os, subprocess, sys
if not os.path.isdir('repo'):
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO,'repo'], check=True)
os.chdir('repo')
print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)
!pip -q install 'torch==2.14.0' 'numpy>=2' 'pandas>=2' 2>/dev/null || pip -q install torch
import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)

In [ ]:
# 2 · ensure the OOF target is present (commit it, or upload here)
import os
p = 'models/stage10/oof_preds.csv'
if not os.path.exists(p):
    os.makedirs('models/stage10', exist_ok=True)
    from google.colab import files
    up = files.upload()                      # pick oof_preds.csv
    name = next(iter(up))
    if name != 'oof_preds.csv':
        os.rename(name, p)
import pandas as pd
d = pd.read_csv(p)
print(f'oof_preds.csv: {len(d):,} rows, seasons {sorted(d.season.unique())}')

In [ ]:
# 3 · re-verify leakage on Colab (gates 5–6)
!python tests/test_stage10_sequence.py

In [ ]:
# 4 · walk-forward pretrain (5 folds) + shared-vs-position-net ablation
!python pipeline/stage10_train.py --pretrain --ablation

In [ ]:
# 5 · NLL curves + gate 2 / gate 3
import json, matplotlib.pyplot as plt
c = json.load(open('models/stage10/stage10_calibration.json'))
fig, ax = plt.subplots(1, len(c['folds']), figsize=(4*len(c['folds']), 3.2), sharey=False)
if len(c['folds']) == 1: ax = [ax]
for a, f in zip(ax, c['folds']):
    ep = [x['epoch'] for x in f['curve']]
    a.plot(ep, [x['train_nll'] for x in f['curve']], label='train')
    a.plot(ep, [x['val_nll'] for x in f['curve']], label='val')
    a.set_title(f"val {f['val_season']}  best={f['best_val_nll']:.3f}")
    a.set_xlabel('epoch'); a.legend(fontsize=8)
ax[0].set_ylabel('Gaussian NLL'); plt.tight_layout(); plt.show()

import pandas as pd
g = pd.DataFrame(c['gate_rows'])
print('\nGATE 2 (MAE) + GATE 3 (q90 coverage):')
print(g[['fold','position','n','mae_gbm','mae_gated','delta_gated','ci_lo','ci_hi','q90_coverage','regression']]
      .to_string(index=False))
reg = g[g.regression]
print('\nRED LINE:', 'CLEAN' if reg.empty else f'{len(reg)} regressions ->\n{reg}')
print('mean ALL q90 coverage:', round(g[g.position=="ALL"].q90_coverage.mean(),3), '(target 0.88–0.92)')
if c.get('ablation'):
    ab = pd.DataFrame(c['ablation'])
    ab['shared_beats_posnet'] = ab.shared_mae <= ab.posnet_mae
    print('\nABLATION shared-trunk vs position-net (MAE, lower better):')
    print(ab.to_string(index=False))

In [ ]:
# 6 · torch vs numpy_forward equivalence (must be < 1e-4)
!python pipeline/stage10_train.py --check-numpy

In [ ]:
# 7 · package checkpoints for the dev machine
import shutil, glob
os.makedirs('/content/out', exist_ok=True)
for f in glob.glob('models/stage10/pretrain_*') + glob.glob('models/stage10/stage10_*.json'):
    shutil.copy(f, '/content/out/')
shutil.make_archive('/content/stage10_checkpoints', 'zip', '/content/out')
from google.colab import files
files.download('/content/stage10_checkpoints.zip')
print('unzip into models/stage10/ on the dev machine')